In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("merge_key", "", "Merge Key")
dbutils.widgets.text("load_type", "", "Load Type")

import time
from datetime import datetime

_run_start = time.time()

# ---------- Parameters ----------
landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys") or dbutils.widgets.get("merge_key")
load_type = dbutils.widgets.get("load_type")

print("=" * 70)
print(f"[START] bronze_to_silver started at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"[PARAMS] table_name        = '{table_name}'")
print(f"[PARAMS] landing_timestamp = '{landing_timestamp}'")
print(f"[PARAMS] load_type         = '{load_type}'")
print(f"[PARAMS] merge_keys        = '{merge_keys}'")
print("=" * 70)

if not table_name:
    print("[WARN] table_name is empty")
if not landing_timestamp:
    print("[WARN] landing_timestamp is empty - no bronze rows will match")
if not load_type:
    print("[WARN] load_type is empty - the write step will fail")

In [0]:
bronze_table = f"adb_healthcare.bronze.{table_name}"
print(f"[BRONZE] Reading batch from {bronze_table}")

bronze_query = f"""
SELECT *
FROM {bronze_table}
WHERE landing_timestamp = '{landing_timestamp}'
"""
print("[BRONZE] Query prepared:")
print("-" * 70)
print(bronze_query.strip())
print("-" * 70)

bronze_df = spark.sql(bronze_query)

bronze_count = bronze_df.count()
print(f"[BRONZE] Rows in batch : {bronze_count}")
print(f"[BRONZE] Columns       : {bronze_df.columns}")
if bronze_count == 0:
    # No new batch (e.g. ADF copied no files). Stop here so a FULL load can't overwrite silver with an empty batch.
    print(f"[SKIP] No rows found in {bronze_table} for landing_timestamp '{landing_timestamp}' - nothing new to load. Silver left unchanged.")
    print(f"[END] bronze_to_silver finished for table '{table_name}' in {time.time() - _run_start:.1f}s")
    print("=" * 70)
    dbutils.notebook.exit("0")


In [0]:
from pyspark.sql.functions import lit, current_timestamp
updated_df=bronze_df.withColumn("insert_timestamp", current_timestamp()).withColumn("update_timestamp", current_timestamp())
print("[TRANSFORM] Set insert_timestamp and update_timestamp to current_timestamp()")
print(f"[TRANSFORM] Output columns: {updated_df.columns}")

In [0]:
#in silver we keep only the final updated version of the record
target_table = f"adb_healthcare.silver.{table_name}"
source_view = f"src_{table_name}_batch"
load_type_normalized = load_type.strip().upper()

print(f"[WRITE] Target table : {target_table}")
print(f"[WRITE] Load type    : {load_type_normalized or '<empty>'}")

if load_type_normalized not in ("FULL", "APPEND", "MERGE"):
    print(f"[ERROR] Unsupported load_type '{load_type}'")
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected one of: FULL, APPEND, MERGE")

target_exists_before = spark.catalog.tableExists(target_table)
target_count_before = spark.read.table(target_table).count() if target_exists_before else 0
print(f"[WRITE] Target exists      : {target_exists_before}")
print(f"[WRITE] Target rows before : {target_count_before}")

cols = updated_df.columns
col_list = ", ".join(f"`{c}`" for c in cols)

def run_sql(step, sql_text):
    """Print the query, execute it, and print how long it took."""
    print(f"[{step}] Query prepared:")
    print("-" * 70)
    print(sql_text.strip())
    print("-" * 70)
    t0 = time.time()
    result = spark.sql(sql_text)
    print(f"[{step}] Query finished in {time.time() - t0:.1f}s")
    return result

if load_type_normalized == "MERGE":
    # Merge: insert new records, update existing ones based on merge keys
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    print(f"[MERGE] Merge keys: {keys}")
    if not keys:
        raise ValueError("merge_keys must be provided (comma-separated) when load_type is MERGE")

    missing_keys = [k for k in keys if k not in cols]
    if missing_keys:
        raise ValueError(f"Merge keys not found in source data: {missing_keys}")

    # A merge fails if multiple source rows match the same target row, so keep one row per key
    source_df = updated_df.dropDuplicates(keys)
    source_count = source_df.count()
    duplicates_removed = updated_df.count() - source_count
    print(f"[MERGE] Source rows after de-duplication on keys: {source_count}")
    if duplicates_removed > 0:
        print(f"[WARN] Removed {duplicates_removed} duplicate rows on merge keys")
else:
    source_df = updated_df

# Register the prepared batch as a temp view so it can be used in SQL
source_df.createOrReplaceTempView(source_view)
print(f"[WRITE] Registered batch as temp view `{source_view}`")

if load_type_normalized == "FULL" or (load_type_normalized in ("MERGE", "APPEND") and not target_exists_before):
    # Full load (or first MERGE/APPEND run): replace / create the target table from the current batch
    if load_type_normalized in ("MERGE", "APPEND"):
        print(f"[{load_type_normalized}] Target does not exist - creating it from the batch")
    else:
        print("[FULL] Overwriting target table with the current batch")
    write_query = f"""
CREATE OR REPLACE TABLE {target_table} AS
SELECT {col_list}
FROM {source_view}
"""
    run_sql(load_type_normalized, write_query)

elif load_type_normalized == "APPEND":
    # Append: add the current batch to the target table (columns matched by name)
    print("[APPEND] Appending current batch to target table")
    write_query = f"""
INSERT INTO {target_table} BY NAME
SELECT {col_list}
FROM {source_view}
"""
    run_sql("APPEND", write_query)

else:
    # MERGE into existing table
    merge_condition = "\n   AND ".join(f"t.`{k}` <=> s.`{k}`" for k in keys)
    # On update, keep the original insert_timestamp and refresh everything else
    update_set = ",\n    ".join(f"t.`{c}` = s.`{c}`" for c in cols if c != "insert_timestamp")
    insert_values = ", ".join(f"s.`{c}`" for c in cols)
    write_query = f"""
MERGE INTO {target_table} AS t
USING {source_view} AS s
    ON {merge_condition}
WHEN MATCHED THEN UPDATE SET
    {update_set}
WHEN NOT MATCHED THEN INSERT ({col_list})
    VALUES ({insert_values})
"""
    merge_result = run_sql("MERGE", write_query).collect()
    if merge_result:
        m = merge_result[0].asDict()
        print(f"[MERGE] Rows affected : {m.get('num_affected_rows')}")
        print(f"[MERGE] Rows updated  : {m.get('num_updated_rows')}")
        print(f"[MERGE] Rows inserted : {m.get('num_inserted_rows')}")
        print(f"[MERGE] Rows deleted  : {m.get('num_deleted_rows')}")

target_count_after = spark.read.table(target_table).count()
print(f"[WRITE] Target rows after  : {target_count_after}")
print(f"[WRITE] Net row change     : {target_count_after - target_count_before}")

last_op = spark.sql(f"DESCRIBE HISTORY {target_table} LIMIT 1").select("version", "timestamp", "operation").collect()[0]
print(f"[WRITE] Delta version {last_op['version']} | {last_op['operation']} | {last_op['timestamp']}")
print(f"[WRITE] {load_type_normalized} load to {target_table} completed successfully")

In [0]:
display(updated_df)

In [0]:
records_count = updated_df.count()
print(f"[EXIT] Returning records_count = {records_count} to caller")
print(f"[END] bronze_to_silver finished for table '{table_name}' in {time.time() - _run_start:.1f}s")
print("=" * 70)
dbutils.notebook.exit(str(records_count))

In [0]:
display(spark.sql("""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT lab_result_id) AS distinct_ids,
       COUNT(DISTINCT landing_timestamp) AS batches,
       MIN(test_date) AS min_test_date,
       MAX(test_date) AS max_test_date
FROM adb_healthcare.silver.lab_results
"""))
display(spark.table("adb_healthcare.silver.lab_results").orderBy("lab_result_id").limit(20))